# ClinHallu: Colab pipeline

Runtime → Change runtime type → GPU. Put the `clinhallu` folder in MyDrive and the three data files in `data/raw/`: `train.jsonl`, `val.jsonl`, `eval_data.jsonl`.

Every command resumes automatically. If the session drops, re-run the setup cell and then the same cell you were on.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/clinhallu
%pip install -q -r requirements/colab.txt
%pip install -q -e . --no-deps

Restart the runtime if Colab asks, then run the cell above again before continuing. Keep Transformers at the version in `requirements/colab.txt` (4.57.6).

## 2. Check environment and data

In [ ]:
!python scripts/check_environment.py
!clinhallu validate --config configs/conditions/c5.yaml --seed 13

## 3. OpenRouter API key

In [ ]:
import os, getpass
# Or use a Colab secret:
# from google.colab import userdata
# os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
os.environ['OPENROUTER_API_KEY'] = getpass.getpass('OpenRouter API key: ')

## 4. Preview (no training, no paid requests)

In [ ]:
!clinhallu generate --config configs/generation/shared_five.yaml --dry-run
!clinhallu pipeline --condition c5 --seed 13 --dry-run

## 5. Run the pipeline (seed 13)

Trains the detector (C5), runs ablations B–F, generates the five shared answers, runs the local baselines (TF-IDF, semantic similarity, zero-shot NLI, SelfCheckGPT-NLI, self-consistency), and builds reports.

In [ ]:
!clinhallu pipeline --condition c5 --seed 13

## 6. Other seeds (42 and 77)

Only training and ablations are repeated; answers and baselines do not depend on the seed.

In [ ]:
!clinhallu pipeline --condition c5 --seed 42 --stages model ablations report
!clinhallu pipeline --condition c5 --seed 77 --stages model ablations report

## 7. LLM judges (paid)

Run one at a time.

In [ ]:
!clinhallu baseline run --only openrouter_gpt_5_mini
!clinhallu baseline run --only openrouter_deepseek_chat
!clinhallu baseline run --only openrouter_llama_3_3_70b_instruct
!clinhallu baseline run --only openrouter_qwen3_32b

## 8. Build final results

In [ ]:
!clinhallu baseline summarize
!clinhallu report build
!python scripts/verify_results.py artifacts/adaptation/c5/seed_13

Tables are in `artifacts/reports/tables/`, figures in `artifacts/reports/figures/`, summaries in `artifacts/reports/summaries/`.

## Resume and troubleshooting

Check what is finished:

In [ ]:
!clinhallu status

- Do not delete `checkpoints/`, the `*_ledger.json` files, `artifacts/pipeline/`, or any `api_cache` folder. These let a run continue.
- To redo a stage: add `--force-stage <name>`. To redo everything: add `--no-resume`.
- To run only one condition (C1–C4, or C5 alone): `!clinhallu condition c1 --seed 13`.